# Week 7: Build a Text Classifier

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: Spam or not spam

"CONGRATULATIONS you won a FREE prize click now." Your email app sorted that into spam before you saw it. Nobody wrote a rule for that exact sentence. Today you will build the same kind of model, trained on examples.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

## Teach 1: Turning words into numbers

Models only understand numbers. `CountVectorizer` builds a **vocabulary** of every word it sees, then turns each sentence into counts: how many times each word appears. This is called a **bag of words**.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

tiny = ["the game is fun", "the game is boring", "fun fun fun"]
vec = CountVectorizer()
counts = vec.fit_transform(tiny)

print("Vocabulary:", vec.get_feature_names_out())
print(pd.DataFrame(counts.toarray(), columns=vec.get_feature_names_out()))

## Teach 2: Train the classifier

**Naive Bayes** learns how likely each word is in a positive review versus a negative one, then adds up the evidence. It is fast and works well on text.

**Concept checkpoint:** before running, predict which word the model will think is the most "negative" in our reviews.

In [ ]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

reviews = load("game_reviews.csv")
X_text_train, X_text_test, y_train, y_test = train_test_split(
    reviews["review"], reviews["label"], test_size=0.25, random_state=3)

vec = CountVectorizer()
X_train = vec.fit_transform(X_text_train)      # learn vocabulary AND transform
X_test = vec.transform(X_text_test)            # only transform (same vocabulary)

clf = MultinomialNB()
clf.fit(X_train, y_train)
print("Test accuracy:", round(accuracy_score(y_test, clf.predict(X_test)) * 100, 1), "%")

In [ ]:
# Which words carry the most evidence?
import numpy as np
words = vec.get_feature_names_out()
neg_idx = list(clf.classes_).index("negative")
pos_idx = list(clf.classes_).index("positive")
evidence = clf.feature_log_prob_[pos_idx] - clf.feature_log_prob_[neg_idx]
order = np.argsort(evidence)
print("Most negative words:", list(words[order[:8]]))
print("Most positive words:", list(words[order[-8:]]))

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Write three new reviews and ask the model to classify them. Use `vec.transform` then `clf.predict`.

### Medium
Add ten labeled sentences of your own to the training data (five positive, five negative) about a topic you know: a movie, a sport, a food. Retrain and test on three new sentences about that topic.

### Spicy
The bag of words cannot tell "not fun" from "fun". Rebuild the vectorizer with `ngram_range=(1, 2)` so it also counts word pairs. Compare accuracy. Then test "this is not fun at all" on both models.

In [ ]:
# MILD: classify new reviews
new_reviews = [
    "The levels are creative and I had a blast",
    "It crashed and the controls are awful",
    "Boring story but the music is great",
]
predictions = clf.predict(vec.transform(new_reviews))
for r, p in zip(new_reviews, predictions):
    print(p.upper(), ":", r)

In [ ]:
# MEDIUM: add your own training data
my_texts = [
    "The movie had a thrilling plot and stunning effects",
    "I loved the soundtrack and the acting was superb",
    "Best movie night ever, we laughed the whole time",
    "The hero was brave and the ending was satisfying",
    "Beautiful scenes and a clever twist",
    "The movie was slow and the plot made no sense",
    "Terrible acting and a predictable ending",
    "I fell asleep halfway through, so dull",
    "The effects looked cheap and the dialogue was cringe",
    "A waste of two hours, do not bother",
]
my_labels = ["positive"] * 5 + ["negative"] * 5
all_texts = list(reviews["review"]) + my_texts
all_labels = list(reviews["label"]) + my_labels

vec2 = CountVectorizer()
clf2 = MultinomialNB().fit(vec2.fit_transform(all_texts), all_labels)
tests = ["The plot was thrilling and the acting superb",
         "Predictable and dull, I nearly fell asleep",
         "Stunning effects but a slow story"]
for t, p in zip(tests, clf2.predict(vec2.transform(tests))):
    print(p.upper(), ":", t)

In [ ]:
# SPICY: word pairs (bigrams)
vec_pairs = CountVectorizer(ngram_range=(1, 2))
Xp_train = vec_pairs.fit_transform(X_text_train)
Xp_test = vec_pairs.transform(X_text_test)
clf_pairs = MultinomialNB().fit(Xp_train, y_train)
print("Single words accuracy:", round(accuracy_score(y_test, clf.predict(X_test)) * 100, 1), "%")
print("Word pairs accuracy:  ", round(accuracy_score(y_test, clf_pairs.predict(Xp_test)) * 100, 1), "%")

s = ["this is not fun at all"]
print("Single words says:", clf.predict(vec.transform(s))[0])
print("Word pairs says:  ", clf_pairs.predict(vec_pairs.transform(s))[0])
print("Vocabulary size grew from", len(vec.get_feature_names_out()), "to", len(vec_pairs.get_feature_names_out()))

## Extra activities (if you finish early)

- Print the full vocabulary size. How many words did the model learn from 80 reviews?
- Try a review written in ALL CAPS. Does the model care? (Hint: `CountVectorizer` lowercases by default.)
- **Colab only:** run `from transformers import pipeline; pipeline("sentiment-analysis")("this is not fun at all")` to see a model trained on millions of sentences. Compare it to yours.

## Reflection

- Why does the model need to see examples of both classes?
- What is one thing a bag of words throws away about a sentence?